# Training, provenance and exact recovery
Purpose: reusable short-run orchestration, finite-gradient step, EMA, atomic checkpoints, and certificate for exact interrupted continuation. Inputs: versioned config, model package, explicit output paths. Outputs: metrics and resumable checkpoints. Runtime on import: <1 second; recovery test: typically <1 minute. All functions are definitions; importing runs no training.

Checkpoints belong in `checkpoints/<run_id>/`. Every synthetic step is saved, including model/student/predictor/teacher, optimizer, constant scheduler, explicit disabled scaler, Python/NumPy/CPU/CUDA/data-mask RNGs, synthetic-stream sample position, configuration, source hashes and metrics. Latest two plus best are retained. Resume: call `run_synthetic` with the identical config, source hashes and run directory; invalid manifests/hashes fail closed. A notebook-level `FMM_JEPA_RESUME_RUN_ID` selects the run. Exact resume is tested on CPU float32; a different hardware/software stack is not guaranteed bitwise identical.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
from pathlib import Path
from contextlib import contextmanager
import copy, csv, datetime, fcntl, hashlib, json, os, random, tempfile, time, uuid
import numpy as np
import torch
from fmm_jepa.models import JEPA
from fmm_jepa.data import sample_masks, synthetic_images

ARTIFACT_SCHEMA = "fmm-jepa-artifact/v1"
CONFIG_SCHEMA = "fmm-jepa-config/v1"

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        with temporary.open('w') as handle:
            json.dump(value, handle, indent=2, allow_nan=False); handle.write('\n')
            handle.flush(); os.fsync(handle.fileno())
        os.replace(temporary, path)
        descriptor = os.open(path.parent, os.O_DIRECTORY)
        try: os.fsync(descriptor)
        finally: os.close(descriptor)
    finally:
        if temporary.exists(): temporary.unlink()

def digest_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def definition_hashes(root):
    result = {}
    for path in sorted((Path(root) / 'fmm_jepa').glob('*.ipynb')):
        notebook = json.loads(path.read_text())
        sources = [''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
        result[str(path.relative_to(root))] = hashlib.sha256('\n'.join(sources).encode()).hexdigest()
    return result

def config_hash(config):
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def capture_rng(generator):
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch_cpu': torch.get_rng_state(), 'torch_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            'data_mask_generator': generator.get_state()}

def restore_rng(state, generator):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu']); generator.set_state(state['data_mask_generator'])
    if state['torch_cuda']:
        if not torch.cuda.is_available() or len(state['torch_cuda']) != torch.cuda.device_count():
            raise RuntimeError('Exact resume requires the same visible CUDA device count')
        torch.cuda.set_rng_state_all(state['torch_cuda'])

def execution_compatibility(config):
    import platform
    device=str(config['device'])
    return {'python':platform.python_version(),'torch':str(torch.__version__),'cuda_build':torch.version.cuda,
            'device':device,'dtype':config['dtype'],'cpu_threads':torch.get_num_threads(),
            'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),
            'machine':platform.machine(),'device_name':torch.cuda.get_device_name(device) if device.startswith('cuda') else platform.processor(),
            'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,'tf32_cudnn':torch.backends.cudnn.allow_tf32}

def make_training_state(config):
    if config['schema_version'] != CONFIG_SCHEMA: raise ValueError('Configuration schema mismatch')
    if config['dtype'] != 'float32': raise ValueError('Training correctness currently supports float32 only')
    if config['world_size'] != 1 or config['gradient_accumulation'] != 1:
        raise NotImplementedError('Distributed/accumulated training is a future protocol; do not silently ignore it')
    seed_everything(config['seed'])
    model = JEPA(config['model']).to(config['device'])
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=config['learning_rate'], weight_decay=0.01)
    # Constant schedule is explicit; its state is checkpointed even though the LR stays fixed.
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 1.0)
    generator = torch.Generator(device='cpu').manual_seed(config['seed'] + 1)
    return model, optimizer, scheduler, generator

def move_masks(value, device):
    if torch.is_tensor(value): return value.to(device)
    if isinstance(value, list): return [move_masks(x, device) for x in value]
    if isinstance(value, dict): return {k:move_masks(v,device) for k,v in value.items()}
    return value

def synthetic_step(model, optimizer, scheduler, generator, config, step):
    model.train()
    images = synthetic_images(config['batch_size'], config['image_height'], config['image_width'], generator).to(config['device'])
    patch = config['model']['patch_size']; grid = (config['image_height']//patch, config['image_width']//patch)
    masks = sample_masks(grid, config['batch_size'], generator=generator, **config['mask'])
    masks = move_masks(masks, config['device'])
    optimizer.zero_grad(set_to_none=True)
    output = model(images, masks['context_indices'], masks['context_valid'], masks['target_indices'], masks['target_valid'])
    loss = output['loss']
    if not bool(torch.isfinite(loss)): raise FloatingPointError('Nonfinite loss; optimizer and EMA not updated')
    loss.backward()
    gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not gradients or not all(bool(torch.isfinite(g).all()) for g in gradients):
        raise FloatingPointError('Nonfinite or absent gradients; optimizer and EMA not updated')
    gradient_norm = float(torch.sqrt(sum(g.detach().square().sum() for g in gradients)))
    optimizer.step(); scheduler.step(); model.update_teacher(config['ema_momentum'])
    if not all(bool(torch.isfinite(p).all()) for p in model.parameters()): raise FloatingPointError('Nonfinite updated parameters')
    import psutil
    row = {'step': int(step), 'loss': float(loss.detach()), 'gradient_norm':gradient_norm,
           'visible_tokens_mean':float(masks['context_valid'].sum(1).float().mean()),
           'process_rss_bytes':psutil.Process().memory_info().rss,
           'cuda_allocated_bytes':torch.cuda.memory_allocated(config['device']) if str(config['device']).startswith('cuda') else 0}
    row.update({key:float(value) for key,value in output['metrics'].items()})
    return row

@contextmanager
def run_writer(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir); checkpoint_dir.mkdir(parents=True, exist_ok=True)
    with (checkpoint_dir / '.writer.lock').open('a+') as handle:
        try: fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError: raise RuntimeError(f'Another writer owns {checkpoint_dir}')
        try: yield
        finally: fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

def save_checkpoint(checkpoint_dir, model, optimizer, scheduler, generator, config, metrics, source_hashes):
    """Call while holding run_writer; publish fsynced checkpoint before atomic manifest."""
    directory = Path(checkpoint_dir); directory.mkdir(parents=True, exist_ok=True)
    step = int(metrics[-1]['step']) if metrics else 0
    old = json.loads((directory/'manifest.json').read_text()) if (directory/'manifest.json').exists() else None
    if old is not None:
        # Reject invalid state before writing over evidence, including stale/changed source.
        validate_manifest(directory, config, source_hashes)
        if step <= old['global_step']: raise ValueError('Checkpoint steps must increase; resume latest state first')
    payload = {'schema_version':ARTIFACT_SCHEMA,'purpose':'synthetic_smoke_test','global_step':step,
               'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
               'scaler':None,'scaler_policy':'disabled: float32 correctness','rng':capture_rng(generator),
               'sampler':{'mode':'synthetic_stream','next_step':step+1,'samples_seen':step*config['batch_size']},
               'distributed':{'world_size':1,'rank':0,'gradient_accumulation':1,'microstep':0},'replay':None,
               'execution_compatibility':execution_compatibility(config),
               'config':copy.deepcopy(config),'config_hash':config_hash(config),'source_hashes':source_hashes,'metrics':copy.deepcopy(metrics)}
    filename = f'step_{step:06d}.pt'; target = directory/filename
    temporary = directory/(filename+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with temporary.open('wb') as handle:
            torch.save(payload,handle); handle.flush(); os.fsync(handle.fileno())
        os.replace(temporary,target)
    finally:
        if temporary.exists(): temporary.unlink()
    score = float(metrics[-1]['loss']) if metrics else None
    entry = {'file':filename,'sha256':digest_file(target),'step':step,'loss':score}
    entries = ([] if old is None else old['checkpoints']) + [entry]
    best = min(entries,key=lambda item:float('inf') if item['loss'] is None else item['loss'])
    latest_two = sorted(entries,key=lambda item:item['step'])[-2:]
    kept = {item['file']:item for item in latest_two+[best]}
    manifest = {'schema_version':ARTIFACT_SCHEMA,'purpose':'synthetic_smoke_test','latest':filename,'best':best['file'],
                'global_step':step,'latest_loss':score,'config_hash':config_hash(config),'source_hashes':source_hashes,
                'resume_notebook':'notebooks/00_FMM_JEPA_Core_Theory_and_Verification.ipynb',
                'resume_cell':'run-synthetic-smoke','checkpoints':sorted(kept.values(),key=lambda item:item['step'])}
    atomic_json(directory/'manifest.json',manifest)
    # Delete only superseded entries from the old, verified manifest after publication.
    for item in entries:
        if item['file'] not in kept: (directory/item['file']).unlink(missing_ok=True)
    return manifest

def validate_manifest(directory, config=None, source_hashes=None):
    directory = Path(directory)
    manifest = json.loads((directory/'manifest.json').read_text())
    if manifest['schema_version'] != ARTIFACT_SCHEMA or manifest['purpose'] != 'synthetic_smoke_test': raise ValueError('Checkpoint schema/purpose mismatch')
    if config is not None and manifest['config_hash'] != config_hash(config): raise ValueError('Checkpoint configuration mismatch')
    if source_hashes is not None and manifest['source_hashes'] != source_hashes: raise ValueError('Exported source changed; incompatible exact resume')
    entries = manifest['checkpoints']
    if not entries or manifest['latest'] not in {e['file'] for e in entries} or manifest['best'] not in {e['file'] for e in entries}: raise ValueError('Malformed checkpoint manifest')
    for entry in entries:
        if Path(entry['file']).name != entry['file']: raise ValueError('Invalid checkpoint filename')
        if digest_file(directory/entry['file']) != entry['sha256']: raise ValueError(f"Checkpoint hash mismatch: {entry['file']}")
    latest = next(e for e in entries if e['file']==manifest['latest'])
    if latest['step'] != manifest['global_step'] or latest['step'] != max(e['step'] for e in entries): raise ValueError('Inconsistent latest step')
    return manifest

def restore_checkpoint(directory, model, optimizer, scheduler, generator, config, source_hashes):
    directory = Path(directory); manifest = validate_manifest(directory,config,source_hashes)
    # Only local, self-authored, hash-verified checkpoints are trusted for weights_only=False.
    payload = torch.load(directory/manifest['latest'],map_location='cpu',weights_only=False)
    if payload.get('execution_compatibility') != execution_compatibility(config):
        raise ValueError('Checkpoint software/device/determinism settings differ; exact resume refused')
    if payload['global_step'] != manifest['global_step'] or payload['config_hash'] != manifest['config_hash'] or payload['source_hashes'] != source_hashes:
        raise ValueError('Checkpoint payload/manifest mismatch')
    if payload['metrics'] and payload['metrics'][-1]['loss'] != manifest['latest_loss']: raise ValueError('Resume metric mismatch')
    if payload['sampler']['next_step'] != payload['global_step']+1: raise ValueError('Invalid sampler position')
    model.load_state_dict(payload['model']); optimizer.load_state_dict(payload['optimizer']); scheduler.load_state_dict(payload['scheduler'])
    restore_rng(payload['rng'],generator)
    return payload

def write_csv(path, rows):
    if not rows: raise ValueError('Refusing an empty results table')
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open('w',newline='') as handle:
        writer=csv.DictWriter(handle,fieldnames=keys); writer.writeheader(); writer.writerows(rows)

def run_synthetic(config, checkpoint_dir, source_hashes):
    model, optimizer, scheduler, generator = make_training_state(config)
    metrics=[]; resumed_from=0
    with run_writer(checkpoint_dir):
        manifest_path=Path(checkpoint_dir)/'manifest.json'
        if manifest_path.exists():
            state=restore_checkpoint(checkpoint_dir,model,optimizer,scheduler,generator,config,source_hashes)
            metrics=state['metrics']; resumed_from=state['global_step']
        elif list(Path(checkpoint_dir).glob('step_*.pt')):
            raise RuntimeError('Checkpoints exist without a manifest; manual recovery required, refusing restart')
        for step in range(resumed_from+1,config['steps']+1):
            metrics.append(synthetic_step(model,optimizer,scheduler,generator,config,step))
            # Every step is a stage boundary in this deliberately short synthetic stream.
            save_checkpoint(checkpoint_dir,model,optimizer,scheduler,generator,config,metrics,source_hashes)
    return model, metrics, {'resumed_from_step':resumed_from,'final_step':len(metrics),'manifest':str(manifest_path)}

def parameter_counts(model):
    groups={name:sum(p.numel() for p in getattr(model,name).parameters()) for name in ('context','predictor','target')}
    groups.update(trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                  non_trainable=sum(p.numel() for p in model.parameters() if not p.requires_grad),
                  buffer_elements=sum(b.numel() for b in model.buffers()))
    assert groups['trainable']==groups['context']+groups['predictor']
    assert groups['non_trainable']==groups['target']
    return groups

def _assert_equal_tree(a,b):
    if torch.is_tensor(a): assert torch.equal(a,b), 'Tensor state differs'
    elif isinstance(a,dict):
        assert a.keys()==b.keys()
        for key in a: _assert_equal_tree(a[key],b[key])
    elif isinstance(a,(list,tuple)):
        assert len(a)==len(b)
        for x,y in zip(a,b): _assert_equal_tree(x,y)
    else: assert a==b, f'State differs: {a!r} != {b!r}'

def checkpoint_resume_test(root, config, source_hashes):
    """Independent interrupted/reference trajectories; checks optimizer, scheduler, RNG and outputs."""
    root=Path(root); root.mkdir(parents=True,exist_ok=True)
    cfg=copy.deepcopy(config); cfg.update(steps=4,device='cpu',image_height=32,image_width=32,batch_size=2)
    cfg['model'].update(dim=16,depth=1,predictor_dim=16,heads=4)
    reference=make_training_state(cfg); reference_metrics=[]
    for step in range(1,5): reference_metrics.append(synthetic_step(*reference,cfg,step))
    reference_rng=capture_rng(reference[3])
    interrupted=make_training_state(cfg); interrupted_metrics=[]
    directory=root/('recovery-'+uuid.uuid4().hex[:10])
    with run_writer(directory):
        for step in range(1,3):
            interrupted_metrics.append(synthetic_step(*interrupted,cfg,step))
            save_checkpoint(directory,*interrupted,cfg,interrupted_metrics,source_hashes)
        restored=make_training_state(cfg)
        payload=restore_checkpoint(directory,*restored,cfg,source_hashes)
        assert payload['global_step']==2 and payload['sampler']['next_step']==3
        _assert_equal_tree(interrupted[0].state_dict(),restored[0].state_dict())
        _assert_equal_tree(interrupted[1].state_dict(),restored[1].state_dict())
        _assert_equal_tree(interrupted[2].state_dict(),restored[2].state_dict())
        # Reading RNG and output does not advance the checkpointed data stream.
        probe=torch.linspace(-1,1,2*3*32*32).reshape(2,3,32,32)
        interrupted[0].eval();restored[0].eval()
        with torch.no_grad(): assert torch.equal(interrupted[0].context(probe),restored[0].context(probe))
        metrics=payload['metrics']
        for step in range(3,5):
            metrics.append(synthetic_step(*restored,cfg,step))
            save_checkpoint(directory,*restored,cfg,metrics,source_hashes)
        _assert_equal_tree(reference[0].state_dict(),restored[0].state_dict())
        _assert_equal_tree(reference[1].state_dict(),restored[1].state_dict())
        _assert_equal_tree(reference[2].state_dict(),restored[2].state_dict())
        assert [r['loss'] for r in reference_metrics]==[r['loss'] for r in metrics]
        current_rng=capture_rng(restored[3])
        for key in ('torch_cpu','torch_cuda','data_mask_generator','python'): _assert_equal_tree(reference_rng[key],current_rng[key])
        assert reference_rng['numpy'][0]==current_rng['numpy'][0]
        assert np.array_equal(reference_rng['numpy'][1],current_rng['numpy'][1])
        assert reference_rng['numpy'][2:]==current_rng['numpy'][2:]
        manifest=validate_manifest(directory,cfg,source_hashes)
        assert {3,4}.issubset({entry['step'] for entry in manifest['checkpoints']})
        # Deliberate corruption only in an isolated copy. Original valid checkpoints remain intact.
        import shutil
        corrupt=root/('corrupt-'+uuid.uuid4().hex[:10]);shutil.copytree(directory,corrupt)
        with (corrupt/manifest['latest']).open('ab') as handle: handle.write(b'corruption-test')
        refused=False
        try: validate_manifest(corrupt,cfg,source_hashes)
        except ValueError: refused=True
        assert refused, 'Corrupt checkpoint was accepted'
        shutil.rmtree(corrupt)
    return {'checkpoint_exact_resume':{'status':'PASS','details':{'bitwise_equal':True,'split_step':2,'final_step':4,'optimizer_restored':True,'scheduler_restored':True,'rngs_restored':True,'corruption_refused':True,'directory':str(directory)}}}
